# X2: Hyperparameter Tuning for RL

**Learning Objectives:**
- Master systematic hyperparameter search
- Implement Population-Based Training (PBT)
- Use Bayesian optimization for RL
- Apply Hyperband/ASHA for efficient tuning
- Understand which hyperparameters matter most
- Build AutoML pipelines for RL

**Prerequisites:** All main curriculum, X1 (Deployment)

**Key Insight**: RL is notoriously sensitive to hyperparameters. Good tuning can mean difference between complete failure and SOTA performance.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from collections import deque
import itertools
from typing import Dict, List, Any, Callable
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Critical Hyperparameters in RL

Understanding what to tune is half the battle.

In [ ]:
# Common RL hyperparameter ranges
RL_HYPERPARAMETER_SPACES = {
    # Learning rates (log scale)
    'learning_rate': {
        'type': 'log_uniform',
        'low': 1e-5,
        'high': 1e-2,
        'importance': 'CRITICAL'
    },
    
    # Discount factor (linear, high values)
    'gamma': {
        'type': 'uniform',
        'low': 0.9,
        'high': 0.999,
        'importance': 'HIGH'
    },
    
    # Network architecture
    'hidden_size': {
        'type': 'choice',
        'values': [64, 128, 256, 512],
        'importance': 'MEDIUM'
    },
    
    'num_layers': {
        'type': 'choice',
        'values': [2, 3, 4],
        'importance': 'MEDIUM'
    },
    
    # PPO-specific
    'clip_epsilon': {
        'type': 'uniform',
        'low': 0.1,
        'high': 0.3,
        'importance': 'HIGH'
    },
    
    'gae_lambda': {
        'type': 'uniform',
        'low': 0.9,
        'high': 0.99,
        'importance': 'MEDIUM'
    },
    
    # Training
    'batch_size': {
        'type': 'choice',
        'values': [32, 64, 128, 256, 512],
        'importance': 'MEDIUM'
    },
    
    'n_epochs': {
        'type': 'choice',
        'values': [3, 5, 10],
        'importance': 'MEDIUM'
    },
    
    # Exploration
    'entropy_coef': {
        'type': 'log_uniform',
        'low': 1e-4,
        'high': 1e-1,
        'importance': 'HIGH'
    }
}

def display_hyperparameter_importance():
    """Display hyperparameter importance ranking."""
    print("RL Hyperparameter Importance Ranking:")
    print("=" * 60)
    
    for importance in ['CRITICAL', 'HIGH', 'MEDIUM']:
        print(f"\n{importance}:")
        for name, config in RL_HYPERPARAMETER_SPACES.items():
            if config['importance'] == importance:
                print(f"  • {name}: {config['type']} {config.get('low', '')} - {config.get('high', config.get('values', ''))}")

display_hyperparameter_importance()

print("\n" + "=" * 60)
print("General Tuning Advice:")
print("=" * 60)
print("1. Learning rate: Most critical, tune first on log scale")
print("2. Batch size: Affects stability and GPU utilization")
print("3. Network size: Bigger ≠ better, risk overfitting")
print("4. Gamma: Higher for long-horizon tasks")
print("5. Entropy: Higher for more exploration")

## 2. Grid Search and Random Search

Foundation of hyperparameter optimization.

In [ ]:
class HyperparameterSampler:
    """Sample hyperparameters for search."""
    
    @staticmethod
    def sample(space):
        """Sample single value from space."""
        if space['type'] == 'uniform':
            return np.random.uniform(space['low'], space['high'])
        elif space['type'] == 'log_uniform':
            log_low = np.log(space['low'])
            log_high = np.log(space['high'])
            return np.exp(np.random.uniform(log_low, log_high))
        elif space['type'] == 'choice':
            return np.random.choice(space['values'])
        else:
            raise ValueError(f"Unknown type: {space['type']}")
    
    @staticmethod
    def sample_config(spaces):
        """Sample complete hyperparameter configuration."""
        config = {}
        for name, space in spaces.items():
            config[name] = HyperparameterSampler.sample(space)
        return config

class RandomSearch:
    """Random search hyperparameter optimization."""
    
    def __init__(self, spaces, objective_fn, n_trials=20):
        """
        Args:
            spaces: Dict of hyperparameter spaces
            objective_fn: Function that takes config and returns score
            n_trials: Number of random trials
        """
        self.spaces = spaces
        self.objective_fn = objective_fn
        self.n_trials = n_trials
        
        self.results = []
        self.best_config = None
        self.best_score = float('-inf')
    
    def run(self):
        """Run random search."""
        print(f"Running random search with {self.n_trials} trials...")
        
        for trial in range(self.n_trials):
            # Sample config
            config = HyperparameterSampler.sample_config(self.spaces)
            
            # Evaluate
            score = self.objective_fn(config)
            
            # Track results
            self.results.append({
                'trial': trial,
                'config': config,
                'score': score
            })
            
            # Update best
            if score > self.best_score:
                self.best_score = score
                self.best_config = config
                print(f"  Trial {trial+1}/{self.n_trials}: New best = {score:.2f}")
        
        return self.best_config, self.best_score
    
    def get_results_df(self):
        """Get results as structured data."""
        return self.results

# Example objective function (simplified)
def toy_objective(config):
    """Toy objective: reward depends on hyperparameters.
    
    In practice, this would train an RL agent and return avg reward.
    """
    # Simulate training with config
    lr = config.get('learning_rate', 1e-3)
    gamma = config.get('gamma', 0.99)
    hidden = config.get('hidden_size', 128)
    
    # Fake reward function (in practice: train agent)
    # Optimal: lr=3e-4, gamma=0.99, hidden=256
    reward = 100 - 50 * abs(np.log10(lr) - np.log10(3e-4))
    reward -= 100 * abs(gamma - 0.99)
    reward -= 0.1 * abs(hidden - 256)
    reward += np.random.normal(0, 5)  # Noise
    
    return reward

# Run random search
search_spaces = {
    'learning_rate': RL_HYPERPARAMETER_SPACES['learning_rate'],
    'gamma': RL_HYPERPARAMETER_SPACES['gamma'],
    'hidden_size': RL_HYPERPARAMETER_SPACES['hidden_size']
}

random_search = RandomSearch(search_spaces, toy_objective, n_trials=20)
best_config, best_score = random_search.run()

print(f"\nBest configuration:")
for k, v in best_config.items():
    print(f"  {k}: {v}")
print(f"Best score: {best_score:.2f}")

print("\nRandom search implemented")

## 3. Bayesian Optimization

Smart search using Gaussian Processes.

In [ ]:
class SimpleBayesianOptimization:
    """Simplified Bayesian Optimization for RL.
    
    Uses random sampling with exploration-exploitation balance.
    In practice, use libraries like Optuna or Ax.
    """
    
    def __init__(self, spaces, objective_fn, n_trials=30, 
                 n_random_init=10, exploration_weight=2.0):
        self.spaces = spaces
        self.objective_fn = objective_fn
        self.n_trials = n_trials
        self.n_random_init = n_random_init
        self.exploration_weight = exploration_weight
        
        self.observations = []
        self.best_config = None
        self.best_score = float('-inf')
    
    def acquisition_function(self, config, observations):
        """Upper Confidence Bound (UCB) acquisition.
        
        In practice, use GP-based acquisition.
        This is simplified version.
        """
        if not observations:
            return 0
        
        # Compute "distance" to previous configs
        distances = []
        for obs in observations:
            dist = 0
            for key in config.keys():
                # Normalized distance
                if key in obs['config']:
                    val1 = config[key]
                    val2 = obs['config'][key]
                    
                    # Handle numeric vs categorical
                    if isinstance(val1, (int, float)):
                        # Normalize to [0,1]
                        space = self.spaces[key]
                        if 'low' in space and 'high' in space:
                            range_val = space['high'] - space['low']
                            dist += ((val1 - val2) / range_val) ** 2
                    else:
                        dist += (0 if val1 == val2 else 1)
            
            distances.append((np.sqrt(dist), obs['score']))
        
        # Estimate mean and uncertainty
        # Use k-nearest neighbors
        k = min(5, len(distances))
        distances.sort(key=lambda x: x[0])
        nearest = distances[:k]
        
        mean_score = np.mean([s for _, s in nearest])
        std_score = np.std([s for _, s in nearest]) if len(nearest) > 1 else 1.0
        
        # UCB
        ucb = mean_score + self.exploration_weight * std_score
        return ucb
    
    def run(self):
        """Run Bayesian optimization."""
        print(f"Running Bayesian optimization with {self.n_trials} trials...")
        
        # Random initialization
        print(f"  Phase 1: Random initialization ({self.n_random_init} trials)")
        for trial in range(self.n_random_init):
            config = HyperparameterSampler.sample_config(self.spaces)
            score = self.objective_fn(config)
            
            self.observations.append({
                'trial': trial,
                'config': config,
                'score': score
            })
            
            if score > self.best_score:
                self.best_score = score
                self.best_config = config
                print(f"    Trial {trial+1}: New best = {score:.2f}")
        
        # Bayesian optimization
        print(f"  Phase 2: Bayesian optimization ({self.n_trials - self.n_random_init} trials)")
        for trial in range(self.n_random_init, self.n_trials):
            # Sample candidates and choose best acquisition
            n_candidates = 100
            best_acq = float('-inf')
            best_candidate = None
            
            for _ in range(n_candidates):
                candidate = HyperparameterSampler.sample_config(self.spaces)
                acq = self.acquisition_function(candidate, self.observations)
                
                if acq > best_acq:
                    best_acq = acq
                    best_candidate = candidate
            
            # Evaluate best candidate
            config = best_candidate
            score = self.objective_fn(config)
            
            self.observations.append({
                'trial': trial,
                'config': config,
                'score': score,
                'acquisition': best_acq
            })
            
            if score > self.best_score:
                self.best_score = score
                self.best_config = config
                print(f"    Trial {trial+1}: New best = {score:.2f} (acq={best_acq:.2f})")
        
        return self.best_config, self.best_score

# Run Bayesian optimization
bayes_opt = SimpleBayesianOptimization(search_spaces, toy_objective, 
                                       n_trials=30, n_random_init=10)
best_config_bayes, best_score_bayes = bayes_opt.run()

print(f"\nBayesian Optimization Results:")
print(f"Best configuration:")
for k, v in best_config_bayes.items():
    print(f"  {k}: {v}")
print(f"Best score: {best_score_bayes:.2f}")

# Compare with random search
print(f"\nComparison:")
print(f"  Random Search best: {best_score:.2f}")
print(f"  Bayesian Opt best: {best_score_bayes:.2f}")
print(f"  Improvement: {best_score_bayes - best_score:.2f}")

print("\nBayesian optimization implemented")

## 4. Population-Based Training (PBT)

DeepMind's approach: evolve hyperparameters during training.

In [ ]:
class Agent:
    """Simple agent for PBT demo."""
    
    def __init__(self, config, agent_id):
        self.config = config
        self.agent_id = agent_id
        self.score = 0
        self.step = 0
    
    def train_step(self):
        """Simulate training step."""
        # In practice: actual RL training
        lr = self.config['learning_rate']
        
        # Fake progress (better configs improve faster)
        optimal_lr = 3e-4
        lr_quality = 1.0 - min(abs(np.log10(lr) - np.log10(optimal_lr)), 1.0)
        
        improvement = lr_quality * 0.5 + np.random.normal(0, 0.1)
        self.score += improvement
        self.step += 1
        
        return self.score
    
    def copy_from(self, other_agent):
        """Copy weights and config from other agent."""
        self.config = other_agent.config.copy()
        self.score = other_agent.score
        # In practice: copy neural network weights
    
    def perturb_hyperparameters(self, spaces):
        """Perturb hyperparameters (explore)."""
        for key in self.config.keys():
            if np.random.random() < 0.3:  # 30% chance to perturb each param
                if spaces[key]['type'] in ['uniform', 'log_uniform']:
                    # Multiply by factor
                    factor = np.random.choice([0.8, 1.2])
                    self.config[key] *= factor
                    # Clip to bounds
                    self.config[key] = np.clip(
                        self.config[key],
                        spaces[key]['low'],
                        spaces[key]['high']
                    )
                elif spaces[key]['type'] == 'choice':
                    # Random choice
                    self.config[key] = np.random.choice(spaces[key]['values'])

class PopulationBasedTraining:
    """Population-Based Training for RL hyperparameter optimization."""
    
    def __init__(self, spaces, population_size=10, exploit_percentile=20):
        self.spaces = spaces
        self.population_size = population_size
        self.exploit_percentile = exploit_percentile
        
        # Initialize population
        self.population = []
        for i in range(population_size):
            config = HyperparameterSampler.sample_config(spaces)
            agent = Agent(config, agent_id=i)
            self.population.append(agent)
        
        self.history = []
    
    def exploit_and_explore(self, agent, population):
        """PBT exploit and explore step."""
        # Sort population by score
        sorted_pop = sorted(population, key=lambda a: a.score, reverse=True)
        
        # Check if agent is in bottom percentile
        threshold_idx = int(len(sorted_pop) * self.exploit_percentile / 100)
        bottom_agents = sorted_pop[-threshold_idx:]
        
        if agent in bottom_agents:
            # Exploit: copy from top agent
            top_agent = sorted_pop[0]
            agent.copy_from(top_agent)
            
            # Explore: perturb hyperparameters
            agent.perturb_hyperparameters(self.spaces)
            
            return True  # Exploited
        
        return False  # No exploit
    
    def train(self, steps=100, eval_freq=10):
        """Run PBT training."""
        print(f"Running PBT with population of {self.population_size}...")
        
        for step in range(steps):
            # Train all agents
            for agent in self.population:
                agent.train_step()
            
            # Exploit and explore
            if step % eval_freq == 0 and step > 0:
                exploits = 0
                for agent in self.population:
                    exploited = self.exploit_and_explore(agent, self.population)
                    if exploited:
                        exploits += 1
                
                # Track best
                best_agent = max(self.population, key=lambda a: a.score)
                
                self.history.append({
                    'step': step,
                    'best_score': best_agent.score,
                    'best_config': best_agent.config.copy(),
                    'exploits': exploits
                })
                
                print(f"  Step {step}: Best score = {best_agent.score:.2f}, "
                      f"Exploits = {exploits}")
        
        # Return best agent
        best_agent = max(self.population, key=lambda a: a.score)
        return best_agent.config, best_agent.score

# Run PBT
pbt = PopulationBasedTraining(
    spaces=search_spaces,
    population_size=8,
    exploit_percentile=25
)

best_config_pbt, best_score_pbt = pbt.train(steps=100, eval_freq=10)

print(f"\nPBT Results:")
print(f"Best configuration:")
for k, v in best_config_pbt.items():
    print(f"  {k}: {v}")
print(f"Best score: {best_score_pbt:.2f}")

# Visualize PBT progress
plt.figure(figsize=(10, 4))
steps = [h['step'] for h in pbt.history]
scores = [h['best_score'] for h in pbt.history]
plt.plot(steps, scores, marker='o')
plt.xlabel('Training Steps')
plt.ylabel('Best Score')
plt.title('PBT: Best Score Over Time')
plt.grid(True)
plt.tight_layout()
plt.show()

print("\nPopulation-Based Training implemented")

## 5. Hyperband / ASHA

Efficient early stopping for hyperparameter search.

In [ ]:
class ASHA:
    """Asynchronous Successive Halving Algorithm.
    
    Efficiently allocates resources by early-stopping poor configurations.
    """
    
    def __init__(self, spaces, objective_fn, max_steps=1000, 
                 reduction_factor=3, n_trials=27):
        """
        Args:
            spaces: Hyperparameter spaces
            objective_fn: Function(config, steps) -> score
            max_steps: Maximum training steps per trial
            reduction_factor: How aggressively to prune (typically 3 or 4)
            n_trials: Number of trials to run
        """
        self.spaces = spaces
        self.objective_fn = objective_fn
        self.max_steps = max_steps
        self.reduction_factor = reduction_factor
        self.n_trials = n_trials
        
        # Compute rungs (checkpoints)
        self.rungs = []
        steps = max_steps
        while steps >= 1:
            self.rungs.insert(0, int(steps))
            steps //= reduction_factor
        
        self.results = []
        self.best_config = None
        self.best_score = float('-inf')
    
    def run(self):
        """Run ASHA."""
        print(f"Running ASHA with {self.n_trials} trials")
        print(f"  Rungs (checkpoints): {self.rungs}")
        print(f"  Reduction factor: {self.reduction_factor}")
        
        # Track configs at each rung
        configs_at_rung = {rung: [] for rung in self.rungs}
        
        # Start all trials at first rung
        for trial_id in range(self.n_trials):
            config = HyperparameterSampler.sample_config(self.spaces)
            configs_at_rung[self.rungs[0]].append({
                'trial_id': trial_id,
                'config': config,
                'steps_trained': 0
            })
        
        # Process each rung
        for rung_idx, rung_steps in enumerate(self.rungs):
            print(f"\n  Rung {rung_idx+1}/{len(self.rungs)} (steps={rung_steps}):")
            
            configs = configs_at_rung[rung_steps]
            if not configs:
                continue
            
            # Evaluate all configs at this rung
            evaluated = []
            for entry in configs:
                config = entry['config']
                steps_to_train = rung_steps - entry['steps_trained']
                
                # Train/evaluate
                score = self.objective_fn(config, steps_to_train)
                
                evaluated.append({
                    'trial_id': entry['trial_id'],
                    'config': config,
                    'score': score,
                    'steps_trained': rung_steps
                })
                
                # Track best
                if score > self.best_score:
                    self.best_score = score
                    self.best_config = config
                
                self.results.append({
                    'trial_id': entry['trial_id'],
                    'rung': rung_steps,
                    'config': config,
                    'score': score
                })
            
            print(f"    Evaluated {len(evaluated)} configs")
            
            # Promote top configs to next rung
            if rung_idx < len(self.rungs) - 1:
                next_rung = self.rungs[rung_idx + 1]
                n_promote = max(1, len(evaluated) // self.reduction_factor)
                
                # Sort by score and promote top
                evaluated.sort(key=lambda x: x['score'], reverse=True)
                promoted = evaluated[:n_promote]
                
                configs_at_rung[next_rung].extend(promoted)
                
                print(f"    Promoted {n_promote}/{len(evaluated)} to next rung")
                print(f"    Best score at rung: {promoted[0]['score']:.2f}")
        
        return self.best_config, self.best_score

# Modified objective that depends on training steps
def progressive_objective(config, steps):
    """Objective that improves with more training steps."""
    # Base score from config quality
    base_score = toy_objective(config)
    
    # Improvement from training (saturates)
    training_bonus = 30 * (1 - np.exp(-steps / 300))
    
    return base_score + training_bonus

# Run ASHA
asha = ASHA(
    spaces=search_spaces,
    objective_fn=progressive_objective,
    max_steps=243,  # 3^5
    reduction_factor=3,
    n_trials=27
)

best_config_asha, best_score_asha = asha.run()

print(f"\n{'='*60}")
print("ASHA Results:")
print(f"Best configuration:")
for k, v in best_config_asha.items():
    print(f"  {k}: {v}")
print(f"Best score: {best_score_asha:.2f}")

# Compute total steps used
total_steps = sum(r['rung'] for r in asha.results)
print(f"\nEfficiency:")
print(f"  Total training steps: {total_steps}")
print(f"  Naive (all to max): {asha.n_trials * asha.max_steps}")
print(f"  Savings: {(1 - total_steps / (asha.n_trials * asha.max_steps)) * 100:.1f}%")

print("\nASHA implemented")

## 6. Practical Tuning Workflow

Putting it all together.

In [ ]:
class RLTuningWorkflow:
    """Complete hyperparameter tuning workflow for RL."""
    
    @staticmethod
    def quick_search(spaces, objective_fn, time_budget_trials=10):
        """Quick random search for initial exploration."""
        print("Phase 1: Quick Random Search")
        print("=" * 60)
        
        search = RandomSearch(spaces, objective_fn, n_trials=time_budget_trials)
        best_config, best_score = search.run()
        
        print(f"\nQuick search complete: {best_score:.2f}")
        return best_config, best_score
    
    @staticmethod
    def focused_search(spaces, objective_fn, init_config, n_trials=20):
        """Focused search around promising region."""
        print("\nPhase 2: Focused Bayesian Search")
        print("=" * 60)
        
        bayes = SimpleBayesianOptimization(
            spaces, objective_fn, n_trials=n_trials, n_random_init=5
        )
        best_config, best_score = bayes.run()
        
        print(f"\nFocused search complete: {best_score:.2f}")
        return best_config, best_score
    
    @staticmethod
    def final_validation(config, objective_fn, n_runs=5):
        """Validate best config with multiple runs."""
        print("\nPhase 3: Final Validation")
        print("=" * 60)
        
        scores = []
        for run in range(n_runs):
            score = objective_fn(config)
            scores.append(score)
            print(f"  Run {run+1}/{n_runs}: {score:.2f}")
        
        mean_score = np.mean(scores)
        std_score = np.std(scores)
        
        print(f"\nValidation: {mean_score:.2f} ± {std_score:.2f}")
        return mean_score, std_score
    
    @staticmethod
    def complete_workflow(spaces, objective_fn):
        """Run complete tuning workflow."""
        print("\n" + "=" * 60)
        print("COMPLETE HYPERPARAMETER TUNING WORKFLOW")
        print("=" * 60)
        
        # Phase 1: Quick exploration
        quick_config, quick_score = RLTuningWorkflow.quick_search(
            spaces, objective_fn, time_budget_trials=10
        )
        
        # Phase 2: Focused search
        focused_config, focused_score = RLTuningWorkflow.focused_search(
            spaces, objective_fn, quick_config, n_trials=20
        )
        
        # Phase 3: Validation
        final_mean, final_std = RLTuningWorkflow.final_validation(
            focused_config, objective_fn, n_runs=5
        )
        
        print("\n" + "=" * 60)
        print("FINAL RESULTS")
        print("=" * 60)
        print(f"\nBest configuration:")
        for k, v in focused_config.items():
            print(f"  {k}: {v}")
        print(f"\nValidated performance: {final_mean:.2f} ± {final_std:.2f}")
        print(f"Improvement from initial: {final_mean - quick_score:.2f}")
        
        return focused_config, final_mean, final_std

# Run complete workflow
final_config, final_mean, final_std = RLTuningWorkflow.complete_workflow(
    search_spaces, toy_objective
)

print("\nComplete tuning workflow demonstrated")

## Summary

### Hyperparameter Tuning Methods:

1. **Random Search**
   - Simple, parallelizable
   - Good baseline
   - Often better than grid search

2. **Bayesian Optimization**
   - Sample-efficient
   - Good for expensive evaluations
   - Use Optuna or Ax in practice

3. **Population-Based Training (PBT)**
   - Evolves hyperparameters during training
   - Great for RL (DeepMind's approach)
   - Finds schedules, not just values

4. **ASHA / Hyperband**
   - Early stopping of poor configs
   - Massive compute savings (10-100x)
   - State-of-the-art efficiency

### RL-Specific Considerations:

**Most Critical Hyperparameters:**
1. Learning rate (tune first!)
2. Entropy coefficient (exploration)
3. Batch size (stability)
4. Gamma (horizon)
5. Network architecture

**Tuning Strategy:**
1. Start with literature defaults
2. Quick random search (10-20 trials)
3. Bayesian or PBT (20-50 trials)
4. Validate best config (5-10 runs)

**Common Pitfalls:**
- Not tuning on same distribution as final eval
- Over-tuning on specific seeds
- Ignoring compute budget
- Tuning too many parameters at once

### Tools in Practice:

- **Optuna**: Bayesian optimization
- **Ray Tune**: PBT, ASHA, all methods
- **Weights & Biases**: Tracking and sweeps
- **Ax (Meta)**: Advanced Bayesian optimization

### Next:

**X3**: Debugging RL systems  
**X4**: Real-world applications and case studies